# 강한 연결 요소 (Strongly Connected Component)

`-` 방향 그래프 내에서 서로 도달 가능한 최대 정점 집합

## 식량 창고

- 문제 출처: [정올 7000번](https://jungol.co.kr/problem/7000)

`-` 문제의 정의상 그래프 내 강한 연결 요소 각각에 대해 최소 비용을 채택하면 된다

`-` 사실 $N$이 작아서 SCC를 사용 안 해도 되지만 단계별 문제니까 연습 삼아 풀어보자

In [5]:
def solution():
    N = int(input())
    costs = list(map(int, input().split()))
    graph = [[] for _ in range(N)]
    for u in range(N):
        adj = input().rstrip()
        for v, s in enumerate(adj):
            if s == "1":
                graph[u].append(v)
    sccs, _ = find_strongly_connected_components(graph)
    answer = 0
    for scc in sccs:
        u = min(scc, key=costs.__getitem__)
        answer += costs[u]
    print(answer)


solution()

# input
# 2
# 1 100
# 01
# 10

 2
 1 100
 01
 10


1


`-` SCC 구현 코드는 간결성을 위해 앞으로 생략하겠다

`-` 타잔 알고리즘을 사용해 $O(|V| + |E|)$에 동작하는 SCC 구현체: https://github.com/Jaesu26/python-cp-templates/blob/main/src/scc.py

## 유명한 소

- 문제 출처: [정올 1622번](https://jungol.co.kr/problem/1622)

`-` 그래프를 여러 SCC로 쪼개보자. 각 SCC 내에선 소끼리 서로 유명하다고 생각한다. 문제는 다른 SCC와의 비교이다

`-` 모든 소들이 유명하다고 여겨야 하므로 진출 차수가 $0$인 SCC에 속한 소들만이 가장 유명할 수 있다. 또한 진출 차수가 $0$인 SCC가 둘 이상이면 서로는 유명하지 않다고 생각하므로 단 하나만 존재해야 한다. 따라서 진출 차수가 $0$인 SCC가 단 하나만 존재하면 이것의 크기가 정답이고 아니면 $0$이다

In [49]:
import sys

sys.setrecursionlimit(10**4 + 2)


def solution():
    N, M = map(int, input().split())
    graph = [[] for _ in range(N)]
    for _ in range(M):
        A, B = map(lambda x: int(x) - 1, input().split())
        graph[A].append(B)
    sccs, scc_ids = find_strongly_connected_components(graph)
    outdegrees = [0] * len(sccs)
    for u in range(N):
        i = scc_ids[u]
        for v in graph[u]:
            j = scc_ids[v]
            outdegrees[i] += i != j
    sinks = [d for d in outdegrees if d == 0]
    answer = len(sccs[sinks[0]]) if len(sinks) == 1 else 0
    print(answer)


solution()

# input
# 3 3
# 1 2
# 2 3
# 3 1

 3 3
 1 2
 2 3
 3 1


3


- 정올피셜 SCC 추천 문제 (ID순)

1. [스키장 짓기](https://jungol.co.kr/problem/1028)

2. [경로강화](https://jungol.co.kr/problem/1209)

3. [선인장](https://jungol.co.kr/problem/1845)

4. [현금 인출기(ATM)](https://jungol.co.kr/problem/2423)

5. [도미노들](https://jungol.co.kr/problem/3099)

## 도미노들

- 문제 출처: [정올 3099번](https://jungol.co.kr/problem/3099)

`-` 도미노를 노드로 배치를 간선으로 하여 그래프로 표현하자. 진입 차수가 $0$인 노드는 무조건 손으로 넘어뜨려야 한다. 근데 이러면 끝 아니야?

`-` 처음엔 진입 차수가 $0$인 노드의 개수가 정답인 줄 알았는데 아니었다. 고립된 사이클이 존재한다면 진입 차수는 $0$이 아니지만 손으로 하나를 넘어뜨려야 나머지를 모두 넘어뜨릴 수 있다. 따라서 SCC로부터 condensation graph(각 SCC를 하나의 노드로 취급)를 만든 뒤 이 그래프에서 진입 차수가 $0$인 노드 개수를 세야 올바른 정답을 도출할 수 있다

In [52]:
import sys

sys.setrecursionlimit(10**5 + 2)


def solve_testcase():
    N, M = map(int, input().split())
    graph = [[] for _ in range(N)]
    for _ in range(M):
        x, y = map(lambda x: int(x) - 1, input().split())
        graph[x].append(y)
    sccs, scc_ids = find_strongly_connected_components(graph)
    dag = build_condensation_graph(graph, sccs, scc_ids)
    n_sccs = len(sccs)
    s = set(range(n_sccs))
    for u in range(n_sccs):
        s.difference_update({v for v in dag[u]})
    return len(s)


def solution():
    T = int(input())
    for _ in range(T):
        answer = solve_testcase()
        print(answer)


solution()

# input
# 1
# 3 3
# 1 2
# 2 3
# 3 1

 1
 3 3
 1 2
 2 3
 3 1


1


## 현금 인출기(ATM)

- 문제 출처: [정올 2423번](https://jungol.co.kr/problem/2423)

`-` ATM을 노드로 도로를 간선으로 생각하자. 방문한 노드를 또 방문할 수 있으므로 어떤 노드가 사이클에 속한다면 해당 사이클 내 모든 ATM에서 현금을 인출할 수 있다. 따라서 같은 사이클에 속한다면 사실상 같은 노드이므로 condensation graph로 바꿔서 생각할 수 있다

`-` 문제는 이제 DAG의 source에서 sink 중 한 곳까지 이동할 때 인출할 수 있는 현금의 최대 액수를 구하는 걸로 바뀌었다. 이때 ATM에 $0$원이 든 더미 노드 하나를 만들고 모든 sink와 연결하면 임계 경로 구하는 거랑 동치이다. 이는 위상 정렬 + DP로 간단하게 해결할 수 있다. 단, 위상 정렬 과정에서 소스로부터 도달할 수 없는 노드는 다른 노드의 DP 값을 갱신하면 안 된다 (처음에 이걸 고려 못해서 푸는 데 조금 오래 걸림)

In [63]:
import itertools
import sys

sys.setrecursionlimit(5 * 10**5 + 2)


def topological_sort(graph, source, values):
    n = len(graph)
    indegrees = [0] * n
    for v in itertools.chain.from_iterable(graph):
        indegrees[v] += 1
    stack = [u for u in range(n) if indegrees[u] == 0]
    dp = [-1] * n
    dp[source] = values[source]
    while stack:
        u = stack.pop()
        for v in graph[u]:
            indegrees[v] -= 1
            if indegrees[v] == 0:
                stack.append(v)
            if dp[u] != -1:
                dp[v] = max(dp[u] + values[v], dp[v])
    return dp


def solution():
    N, M = map(int, input().split())
    graph = [[] for _ in range(N)]
    for _ in range(M):
        s, e = map(lambda x: int(x) - 1, input().split())
        graph[s].append(e)
    atms = [int(input()) for _ in range(N)]
    S, P = map(int, input().split())
    restaurants = list(map(lambda x: int(x) - 1, input().split()))
    sccs, scc_ids = find_strongly_connected_components(graph)
    dag = build_condensation_graph(graph, sccs, scc_ids)
    values = [0] * len(sccs)
    for u in range(N):
        values[scc_ids[u]] += atms[u]
    dp = topological_sort(dag, scc_ids[S - 1], values)
    answer = max(dp[scc_ids[r]] for r in restaurants)
    print(answer)


solution()

# input
# 3 2
# 1 2
# 2 3
# 1
# 10
# 100
# 1 1
# 3

 3 2
 1 2
 2 3
 1
 10
 100
 1 1
 3


111


`-` 굳이 더미 노드 안 만들고 condensation graph에서 각 sink가 속한 노드의 DP 값을 순회해도 상관 없어서 이렇게 했다. 그리고 MLE 때문에 PyPy 안 씀

`-` + 여태까지 임계 경로 구하는 문제는 위상 정렬 안 쓰고 싱크에서 재귀 DFS 써가지고 풀었는데 역방향 그래프 안 만들어도 되서 위상 정렬 쓰는 게 더 편하다

## 스키장 짓기

- 문제 출처: [정올 1028번](https://jungol.co.kr/problem/1028)

`-` 스키 이동 조건을 바탕으로 그래프를 만들자. 인접한 두 구역은 단방향으로 연결되거나 양방향으로 연결되거나 둘 중 하나이므로 전체는 하나의 약한 연결 요소이다

`-` 임의의 두 구역을 연결하려면 전체 구역이 하나의 사이클을 이루어야 한다. 이미 사이클을 이루는 구역들은 하나의 구역으로 취급할 수 있으니 전체를 condensation graph로 변환하자. 이 그래프가 하나의 노드로만 표현된다면 이미 전체가 사이클을 이루므로 리프트는 필요 없다. 그렇지 않으면 진입 차수가 $0$인 노드와 진출 차수가 $0$인 노드를 리프트로 연결해서 부분 사이클을 만들 수 있다

`-` 초기 condensation graph에선 위상 정렬이 가능하므로 진입 차수가 $0$인 노드에선 진출 차수가 $0$인 노드로 항상 이동 가능하다. 진입 차수가 $0$인 노드 개수를 $k_{\text{in}}$, 진출 차수가 $0$인 노드 개수를 $k_{\text{out}}$으로 나타내자. 전체 사이클을 만들기 위해 둘을 일대일 매칭 시킨 뒤 남는 건 이미 매칭된 노드와 연결하면 된다. 계산해보면 필요한 리프트의 수는 $\min(k_{\text{in}}, k_{\text{out}}) + \max(k_{\text{in}}, k_{\text{out}}) - \min(k_{\text{in}}, k_{\text{out}}) = \max(k_{\text{in}}, k_{\text{out}})$이다

In [6]:
import sys
from itertools import chain, product

sys.setrecursionlimit(3 * 10**5)


def build_graph(H, n_rows, n_cols):
    graph = [[] for _ in range(n_rows * n_cols)]
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for r, c in product(range(n_rows), range(n_cols)):
        h = H[r][c]
        i = convert(r, c, n_cols)
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_within_grid = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_within_grid or h < H[nr][nc]:
                continue
            graph[i].append(convert(nr, nc, n_cols))
    return graph


def convert(r, c, n_cols):
    return r * n_cols + c


def solution():
    W, L = map(int, input().split())
    H = [list(map(int, input().split())) for _ in range(L)]
    graph = build_graph(H, L, W)
    sccs, scc_ids = find_strongly_connected_components(graph)
    dag = build_condensation_graph(graph, sccs, scc_ids)
    if len(sccs) == 1:
        print(0)
        return
    n_sccs = len(sccs)
    indegrees, outdegrees = [0] * n_sccs, [0] * n_sccs
    for u in chain.from_iterable(dag):
        indegrees[u] += 1
    for u in range(n_sccs):
        outdegrees[u] += len(dag[u])
    k_in = sum(indegrees[u] == 0 for u in range(n_sccs))
    k_out = sum(outdegrees[u] == 0 for u in range(n_sccs))
    answer = max(k_in, k_out)
    print(answer)


solution()

# input
# 3 3
# 1 2 1
# 2 2 2
# 1 2 1

 3 3
 1 2 1
 2 2 2
 1 2 1


4


`-` 이 문제도 MLE 때문에 PyPy 안 쓰고 Python으로 AC 받았다. 비재귀로 바꿔야 할 듯